## STATUS BANNER (updated 2026-07-25, revision 2)

**This notebook has NOT been executed with this code.** All outputs below are empty.

**Revision history:**
- r1: first version, closes ISS-09 (deep baseline).
- r2 (this revision): added a Drive-FUSE-lag fallback to the load cell (`cell-load`).
  You can have `windows_vnext.npz` visibly present in the Drive **web UI** while
  Colab's mounted FUSE filesystem still doesn't see it -- a known Colab quirk
  (`mdc_drive_io.py`'s own docstring already documents this class of issue). The
  cell now: (1) tries a `force_remount=True` and retries, (2) if that still doesn't
  surface the file, prompts you to upload it manually via a file picker
  (`google.colab.files.upload()`) instead of just failing.

**Run order for the full batch** (see `docs/module4_remediation_plan.md`):
1. `mdc_preprocess_vNext_output.ipynb`
2. `mdc_preprocess_vNext_mc_output.ipynb`
3. `mdc_model_vNext_output.ipynb`
4. `mdc_drift_aware_output.ipynb`
5. `mdc_baselines_output.ipynb`  <- this file, run last


# MDC vNext — Deep Baseline (Dense Autoencoder)

**Closes:** ISS-09 (`docs/ISSUES_AND_IMPROVEMENT_PLAN.md` §3.1) — "thin baselines: Isolation Forest only".
**Self-contained:** upload only this notebook. Loads `windows_vnext.npz` from `mdc_preprocess_vNext.ipynb`
and (optionally) `metrics_vnext.json` / `baseline_comparison.json` from the model / drift-aware runs to
build one combined comparison table: **Isolation Forest vs Dense AE vs vNext default vs vNext HPO**,
all evaluated on the identical test windows.

**Why Dense AE, not LSTM-AE:** the WBS's own fallback table (§8.0) lists Dense AE as the acceptable
minimum second baseline ("No time for Dense AE baseline → Isolation Forest only (minimum)"). A
feed-forward autoencoder has far fewer moving parts than an LSTM-AE (no hidden-state handling, no
sequence-length sensitivity), which matters here because this notebook is being handed off unexecuted
(no GPU in the authoring environment) — fewer moving parts means fewer places for an unverified bug
to hide before you run it on Colab.

**Protocol parity with vNext (for a fair comparison):**
- Same `windows_vnext.npz` (same train/val/test windows, same benign-only training set)
- Same auto invert-flip check on val (reconstruction error can come out anti-correlated with "attack")
- Same F1-optimal threshold rule (tuned on val, applied to test)
- Same metric set as the existing Isolation Forest baseline (ROC-AUC, PR-AUC, F1, MCC, precision, recall, FPR)


## 0. Environment

In [ ]:
import sys, subprocess
try:
    import google.colab
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False
if _IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch', 'scikit-learn', 'matplotlib'], check=False)
print(f'In Colab: {_IN_COLAB}')


## 1. Configuration

In [ ]:
from __future__ import annotations
import os, json, gc, random, math
from pathlib import Path
import numpy as np

VERSION = 'vnext'
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Dense AE architecture (deliberately simple -- this is a baseline, not the main model)
HIDDEN_DIM      = 256
BOTTLENECK_DIM  = 64
DROPOUT         = 0.1
BATCH_SIZE      = 64
MAX_EPOCHS      = 60
LR              = 1e-3
WEIGHT_DECAY    = 1e-5
AUC_CHECK_FREQ  = 2
AUTO_SCORE_FLIP = True

NPZ_FILE = f'windows_{VERSION}.npz'

# Same project layout as mdc_model_vNext.ipynb / mdc_drift_aware.ipynb
DRIVE_PROJECT_FOLDER = 'vNEXT_test'
LOCAL_DATA_CACHE = Path('/content/vNEXT_test_local/processed')
LOCAL_RUN_DIR    = Path('/content/vNEXT_test_local/runs')
LOCAL_BASELINE_DIR = Path('/content/vNEXT_test_local/baselines')

if _IN_COLAB:
    DRIVE_ROOT     = Path('/content/drive/MyDrive') / DRIVE_PROJECT_FOLDER
    DRIVE_DATA_DIR = DRIVE_ROOT / 'processed'
    DRIVE_RUN_DIR  = DRIVE_ROOT / 'runs'
    DRIVE_BASELINE_DIR = DRIVE_ROOT / 'baselines'
    DATA_DIR = LOCAL_DATA_CACHE
    RUN_DIR  = LOCAL_RUN_DIR
    BASELINE_DIR = LOCAL_BASELINE_DIR
else:
    DRIVE_ROOT     = Path('../outputs') / DRIVE_PROJECT_FOLDER
    DRIVE_DATA_DIR = DRIVE_ROOT / 'processed'
    DRIVE_RUN_DIR  = DRIVE_ROOT / 'runs'
    DRIVE_BASELINE_DIR = DRIVE_ROOT / 'baselines'
    DATA_DIR = DRIVE_DATA_DIR
    RUN_DIR  = DRIVE_RUN_DIR
    BASELINE_DIR = DRIVE_BASELINE_DIR

for _p in (DATA_DIR, RUN_DIR, BASELINE_DIR, LOCAL_DATA_CACHE, LOCAL_RUN_DIR, LOCAL_BASELINE_DIR,
           DRIVE_DATA_DIR, DRIVE_RUN_DIR, DRIVE_BASELINE_DIR):
    Path(_p).mkdir(parents=True, exist_ok=True)

print('Config loaded')
print(f'  DATA_DIR (use)     : {DATA_DIR}')
print(f'  RUN_DIR  (use)     : {RUN_DIR}')
print(f'  BASELINE_DIR (use) : {BASELINE_DIR}')
print(f'  ARCH: hidden={HIDDEN_DIM} bottleneck={BOTTLENECK_DIM} dropout={DROPOUT}')


## 2. Mount drive + load npz + existing metrics

In [ ]:
# Minimal Drive I/O (subset of notebook/mdc_drive_io.py -- FUSE-safe pull only,
# no push-verification API-bypass helpers since this notebook only needs to
# read existing artifacts and write its own comparison table).
import shutil, time
from datetime import datetime, timezone

def _mount_drive_safe():
    if not _IN_COLAB:
        return
    from google.colab import drive
    mydrive = Path('/content/drive/MyDrive')
    if not mydrive.is_dir():
        drive.mount('/content/drive')
    if not mydrive.is_dir():
        raise RuntimeError('Drive mount failed -- Runtime > Restart runtime, retry, complete auth.')
    print(f'Drive OK: {mydrive}')

def _pull_if_needed(name, local_dir, search_roots):
    local_dir = Path(local_dir); local_dir.mkdir(parents=True, exist_ok=True)
    dest = local_dir / name
    if dest.is_file() and dest.stat().st_size > 0:
        return dest
    for root in search_roots:
        src = Path(root) / name
        if src.is_file():
            shutil.copy2(src, dest)
            print(f'  pulled {name} <- {src}')
            return dest
    return None

if _IN_COLAB:
    _mount_drive_safe()

npz_path = _pull_if_needed(NPZ_FILE, LOCAL_DATA_CACHE, [DRIVE_DATA_DIR, DATA_DIR])

if npz_path is None and _IN_COLAB:
    # Fallback: Drive FUSE can lag behind the web UI (file visible on
    # drive.google.com but not yet visible to the mounted filesystem here --
    # a known Colab quirk, see mdc_drive_io.py's own docstring). Try a
    # force-remount first; if that still doesn't surface it, prompt for a
    # manual upload rather than failing outright.
    print(f'{NPZ_FILE} not visible via Drive mount -- retrying with force_remount ...')
    try:
        from google.colab import drive as _drive
        _drive.mount('/content/drive', force_remount=True)
    except Exception as _e:
        print(f'  remount attempt failed (continuing to upload fallback): {_e}')
    npz_path = _pull_if_needed(NPZ_FILE, LOCAL_DATA_CACHE, [DRIVE_DATA_DIR, DATA_DIR])

if npz_path is None and _IN_COLAB:
    print(f'{NPZ_FILE} still not found via Drive. Falling back to manual upload.')
    print(f'  1. Download {NPZ_FILE} from the Drive web UI to your computer '
          f'(My Drive > vNEXT_test > processed > {NPZ_FILE}).')
    print('  2. Use the file picker that appears below to select it.')
    from google.colab import files
    uploaded = files.upload()
    if NPZ_FILE in uploaded:
        LOCAL_DATA_CACHE.mkdir(parents=True, exist_ok=True)
        npz_path = LOCAL_DATA_CACHE / NPZ_FILE
        npz_path.write_bytes(uploaded[NPZ_FILE])
        print(f'Uploaded and saved -> {npz_path}  ({npz_path.stat().st_size:,} bytes)')
    else:
        npz_path = None
        print(f'Upload did not contain {NPZ_FILE} (got: {list(uploaded.keys())}).')

if npz_path is None:
    raise FileNotFoundError(
        f'{NPZ_FILE} not found under {DRIVE_DATA_DIR} or {DATA_DIR}, remount did not '
        'help, and no valid file was uploaded. Run mdc_preprocess_vNext.ipynb first, '
        'or upload the npz manually.'
    )

z = np.load(npz_path, allow_pickle=True)
X_train = z['X_train'].astype(np.float32)
X_val   = z['X_val'].astype(np.float32)
X_test  = z['X_test'].astype(np.float32)
y_val   = z['y_val'].astype(np.int8)
y_test  = z['y_test'].astype(np.int8)
T, n_features = X_test.shape[1], X_test.shape[2]
input_dim = T * n_features
z.close()
print(f'Loaded {npz_path.name}: train={X_train.shape} val={X_val.shape} test={X_test.shape}  input_dim={input_dim}')

# Optional: pull existing vNext metrics + IF baseline for the combined table (not required to train)
metrics_path = _pull_if_needed('metrics_vnext.json', LOCAL_RUN_DIR, [DRIVE_RUN_DIR, RUN_DIR])
metrics_vnext = json.loads(metrics_path.read_text()) if metrics_path else None
print(f'metrics_vnext.json: {"found" if metrics_vnext else "NOT FOUND (vNext rows will be blank in the table)"}')

if_baseline_path = _pull_if_needed('baseline_comparison.json', LOCAL_RUN_DIR,
                                    [DRIVE_ROOT / 'drift_aware', RUN_DIR])
if_baseline = json.loads(if_baseline_path.read_text()) if if_baseline_path else None
print(f'baseline_comparison.json (IF): {"found" if if_baseline else "NOT FOUND (run mdc_drift_aware.ipynb first for the IF row)"}')


## 3. Tensors / device

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

X_train_flat = X_train.reshape(len(X_train), -1)
X_val_flat   = X_val.reshape(len(X_val), -1)
X_test_flat  = X_test.reshape(len(X_test), -1)

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train_flat).float()),
    batch_size=BATCH_SIZE, shuffle=True, drop_last=False, num_workers=0,
)
print(f'Train batches: {len(train_loader)}  val windows: {len(X_val_flat)}  test windows: {len(X_test_flat)}')


## 4. Dense autoencoder

Flatten each `(T, F)` window to a single `T*F` vector and reconstruct it through a
small bottleneck MLP. Reconstruction MSE is the anomaly score (higher = more anomalous),
same convention as the main vNext transformer AE.

In [ ]:
class DenseAE(nn.Module):
    def __init__(self, input_dim, hidden_dim=HIDDEN_DIM, bottleneck_dim=BOTTLENECK_DIM, dropout=DROPOUT):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, bottleneck_dim), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

def build_dense_ae():
    return DenseAE(input_dim).to(device)

print('DenseAE defined:', f'{input_dim} -> {HIDDEN_DIM} -> {BOTTLENECK_DIM} -> {HIDDEN_DIM} -> {input_dim}')


## 5. Scoring helpers

Same auto invert-flip convention as the main pipeline: reconstruction error can come out
anti-correlated with "attack" depending on what the model learns, so both directions are
checked on val and the better one is used (never assumed).

In [ ]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, matthews_corrcoef, confusion_matrix,
)

@torch.no_grad()
def compute_scores_dense(m, X_flat, bs=256):
    m.eval()
    out = []
    for k in range(0, len(X_flat), bs):
        xb = torch.from_numpy(X_flat[k:k+bs]).float().to(device)
        rb = m(xb)
        err = (rb - xb).pow(2).mean(dim=1)
        out.append(err.cpu().numpy())
        del xb, rb, err
    return np.concatenate(out)

def thresholds_from_val_dense(scores, y):
    order = np.argsort(scores)
    s_sorted = scores[order]
    best_f1, best_thr = -1.0, float(s_sorted[0])
    for thr in np.unique(s_sorted):
        pred = (scores >= thr).astype(int)
        f1 = f1_score(y, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, float(thr)
    return best_thr

def full_eval_dense(scores, y, thr):
    pred = (scores >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        'roc_auc': float(roc_auc_score(y, scores)),
        'pr_auc': float(average_precision_score(y, scores)),
        'f1': float(f1_score(y, pred, zero_division=0)),
        'mcc': float(matthews_corrcoef(y, pred)),
        'precision': float(tp / max(tp + fp, 1)),
        'recall': float(tp / max(tp + fn, 1)),
        'fpr': float(fp / max(fp + tn, 1)),
        'threshold': float(thr),
    }

print('scoring helpers ready')


## 6. Train (benign-only, same convention as vNext)

In [ ]:
model = build_dense_ae()
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_val_auc = -1.0
best_state = None
history = []

y_val_t = y_val  # already int8 array

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for (xb,) in train_loader:
        xb = xb.to(device)
        opt.zero_grad(set_to_none=True)
        rb = model(xb)
        loss = F.mse_loss(rb, xb)
        loss.backward()
        opt.step()
        epoch_loss += loss.item() * len(xb)
    epoch_loss /= len(X_train_flat)

    if epoch % AUC_CHECK_FREQ == 0 or epoch == MAX_EPOCHS:
        s_val_raw = compute_scores_dense(model, X_val_flat)
        auc_raw  = roc_auc_score(y_val_t, s_val_raw)
        auc_flip = roc_auc_score(y_val_t, -s_val_raw)
        auc = max(auc_raw, auc_flip)
        history.append({'epoch': epoch, 'loss': epoch_loss, 'val_auc': auc})
        if auc > best_val_auc:
            best_val_auc = auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f'epoch {epoch:>3}  loss={epoch_loss:.5f}  val_auc={auc:.4f}  best={best_val_auc:.4f}')

if best_state is not None:
    model.load_state_dict(best_state)
print(f'\nTraining done. best_val_auc={best_val_auc:.4f}')


## 7. Evaluate on test (F1-optimal threshold, same protocol as IF baseline)

In [ ]:
s_val_raw  = compute_scores_dense(model, X_val_flat)
s_test_raw = compute_scores_dense(model, X_test_flat)

val_raw  = roc_auc_score(y_val,  s_val_raw)
val_flip = roc_auc_score(y_val, -s_val_raw)
invert = bool(AUTO_SCORE_FLIP and val_flip > val_raw)
print(f'val AUC raw={val_raw:.4f} flip={val_flip:.4f} -> invert={invert}')

s_val  = -s_val_raw  if invert else s_val_raw
s_test = -s_test_raw if invert else s_test_raw

thr = thresholds_from_val_dense(s_val, y_val)
dense_ae_result = full_eval_dense(s_test, y_test, thr)
dense_ae_result['score_inverted'] = invert
dense_ae_result['best_val_auc'] = float(best_val_auc)

print('\nDense AE baseline -- test metrics (F1-optimal threshold):')
for k in ('roc_auc', 'pr_auc', 'f1', 'mcc', 'precision', 'recall', 'fpr'):
    print(f'  {k:<10} {dense_ae_result[k]:.4f}')


## 8. Combined comparison table (ISS-09)

`Isolation Forest` (from `mdc_drift_aware.ipynb`) vs `Dense AE` (this notebook) vs
`vNext default` / `vNext HPO` (from `mdc_model_vNext.ipynb`) -- all on the identical
test windows from `windows_vnext.npz`, so the comparison is apples-to-apples.

In [ ]:
rows = [{'config': 'Dense AE (this notebook)', **{k: dense_ae_result[k] for k in
         ('roc_auc', 'pr_auc', 'f1', 'mcc', 'precision', 'recall', 'fpr')}, 'source': 'mdc_baselines.ipynb'}]

if if_baseline is not None:
    r_if = if_baseline.get('isolation_forest_mean_max', {}).get('metrics', {})
    if r_if:
        rows.append({'config': 'Isolation Forest (mean_max pool)',
                      **{k: r_if.get(k) for k in ('roc_auc', 'pr_auc', 'f1', 'mcc', 'precision', 'recall', 'fpr')},
                      'source': 'mdc_drift_aware.ipynb'})

if metrics_vnext is not None:
    r_def = metrics_vnext.get('test_default', {}).get('f1_optimal', {})
    if r_def:
        rows.append({'config': 'vNext default (Transformer AE)',
                      **{k: r_def.get(k) for k in ('roc_auc', 'pr_auc', 'f1', 'mcc', 'precision', 'recall', 'fpr')},
                      'source': 'mdc_model_vNext.ipynb'})
    r_hpo = metrics_vnext.get('test_hpo', {})
    if r_hpo:
        rows.append({'config': 'vNext HPO best (Transformer AE)',
                      **{k: r_hpo.get(k) for k in ('roc_auc', 'pr_auc', 'f1', 'mcc', 'precision', 'recall', 'fpr')},
                      'source': 'mdc_model_vNext.ipynb'})

import pandas as pd
comparison_df = pd.DataFrame(rows)
print('\n=== Deep + classical baseline comparison (ISS-09) ===')
print(comparison_df.to_string(index=False))

if len(rows) < 4:
    _missing = {'Isolation Forest'} if if_baseline is None else set()
    if metrics_vnext is None:
        _missing |= {'vNext default', 'vNext HPO'}
    print(f'\nNOTE: table is partial -- run these notebooks first to fill it in: {sorted(_missing)}'
          if _missing else '')

out = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'dense_ae': dense_ae_result,
    'comparison_table': comparison_df.to_dict(orient='records'),
    'protocol': {
        'same_windows_vnext_npz': True,
        'threshold_rule': 'f1_optimal_on_val',
        'auto_score_flip': AUTO_SCORE_FLIP,
    },
}
out_json = BASELINE_DIR / 'deep_baseline_comparison.json'
out_csv  = BASELINE_DIR / 'deep_baseline_comparison.csv'
out_json.write_text(json.dumps(out, indent=2))
comparison_df.to_csv(out_csv, index=False)
print(f'\nSaved -> {out_json}')
print(f'Saved -> {out_csv}')


## 9. Push results to Drive (Colab only)

In [ ]:
if _IN_COLAB:
    for name in ('deep_baseline_comparison.json', 'deep_baseline_comparison.csv'):
        src = BASELINE_DIR / name
        if src.is_file():
            dest = DRIVE_BASELINE_DIR / name
            shutil.copy2(src, dest)
            print(f'Pushed {name} -> {dest}')
    print(f'\nDrive UI: My Drive > {DRIVE_PROJECT_FOLDER} > baselines')
else:
    print('Not in Colab -- results saved locally only:', BASELINE_DIR)
